# 01. EDA и симуляция транзакций (PaySim-формат)

Исследование распределения сумм, временной динамики, дисбаланса классов, сценариев мошенничества и графа переводов (NetworkX).

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
from src.data.simulator import load_or_generate_dataset
from src.features.engineering import build_features_chronological

df = load_or_generate_dataset()
print(f"Всего транзакций: {len(df):,} | Доля фрода: {df['isFraud'].mean():.3%}")
df.head()

In [ ]:
# Распределение сценариев мошенничества
df["fraud_scenario"].value_counts()

In [ ]:
# Временная динамика доли мошенничества по дням (дрейф концепции в конце периода)
df["day"] = df["step"] // 24
daily_fraud = df.groupby("day")["isFraud"].mean()
ax = daily_fraud.plot(kind="line", marker="o", figsize=(10, 4), title="Доля мошенничества по дням")
ax.set_ylabel("Fraud Rate")
plt.grid(True)
plt.show()

In [ ]:
# Построение leak-free признаков и графа получателей-дропперов
feat_df, store = build_features_chronological(df, return_store=True)
g = store.build_networkx_graph(top_dest_min_degree=5)
print(f"Узлов в подозрительном графе: {g.number_of_nodes()}, рёбер: {g.number_of_edges()}")
feat_df.groupby("isFraud")[[
    "amount", "balance_drain_ratio", "orig_tx_count_1h",
    "amount_zscore", "dest_in_degree", "device_shared_accounts"
]].mean().T